# Session 23 · Serving BankAssist — an API with memory and streaming
## From a script only you can run, to a service anyone can call

**Last session you wrote BankAssist yourself.** But it lived in a script: only you could use it, from your terminal,
one question at a time, and it forgot everything between runs. Today you put it behind a **web API**. Any program
(a website, a mobile app, a teammate's code) can then talk to it, it remembers each conversation separately, and it
streams its answer word by word.

| Part | What it is |
|---|---|
| **Part A · Read before you build** | Why serve it, the words you need, how today works (two terminals), the rules for ChatGPT |
| **Part B · Build** | Section 1 sets up your file. Sections 2–7 are the six TODOs. Section 8 runs it for real. Section 9 breaks it on purpose. Section 10 is a LiteLLM bridge (S22 → next session). Sections 11–12 check and submit. |
| **Part C · Wrap-up** | Summary, honest limits, interview questions, homework (**including the Docker install you need for Session 23b**) |

**Where you type.** In `s23/app.py` in your work repository, in VS Code, the same way as Session 21. The notebook
explains each step and checks your code after each one.

---
# Part A · Read before you build

## A1. Why this session matters

Nobody uses a bank's assistant by running `python bankassist.py`. They type into a website or an app. That website
sends the question over HTTP to a **server**, and the server sends the answer back. Today you build that server.

Three things change when your code becomes a service:

1. **Other programs call it.** So the request and the reply need a fixed shape, checked on the way in. A bad request
   should be rejected *before* it costs a token.
2. **Many people use it at once.** So each conversation needs its own memory, kept apart. Session 19's `messages`
   list now lives on the server, one list per `session_id`.
3. **People are waiting.** A 6-second answer feels broken; the same answer starting after 1 second, word by word,
   feels fast. That's **streaming**.

**What you learn today, and why it matters at work:**

| You learn to... | It saves you from... |
|---|---|
| Turn functions into endpoints with FastAPI | Writing HTTP handling by hand |
| Describe requests and replies with Pydantic | Paying for (and debugging) garbage requests |
| Keep one memory per session, trimmed to a window | Customers seeing each other's data; costs that grow forever |
| Stream with Server-Sent Events | A UI that looks frozen for six seconds |
| Test an API without starting a server | "It worked on my machine" demos |

**Why interviewers care.** "How would you put this model behind an API?" comes up in almost every GenAI interview.
After today your answer has specifics: FastAPI, Pydantic validation, per-session memory with a window, SSE streaming,
and the honest limits of keeping memory in RAM.

## A2. Where this session fits
```
 S19  memory: the messages list, the sliding window ──► SESSIONS + window()
 S20  tools, raw; Fix B: validate and feed back ──────► run_tool() (given)
 S21  LangChain: YOU built BankAssist ────────────────► PART 1 of today's file (given, finished)
 S22  LiteLLM: one call, fallbacks, cost ─────────────► (bridge in Section 10)
  │
 S23  FastAPI + SSE: YOU serve it ────────────────────► /health · /chat · /chat/stream   (+ LiteLLM bridge)
  │
 S23b Gateway + Docker ──────────────────────────────► the LiteLLM proxy (from S22) and this server, in containers
 MP1  ShopAssist (2 weekend sessions, solo) ─────────► a fresh app: LangChain + gateway + Docker + evidence
 RAG ─────────────────────────────────────────────────► the same server, answering from documents
 Cloud deployment ────────────────────────────────────► the same container, on a cloud
```

## A3. The words you need today
The full table is in `module03/terms_s23.md`. The seven you'll use in the next hour:

- **Endpoint**: one URL plus one method, e.g. `POST /chat`. In FastAPI it's a normal function with a decorator.
- **FastAPI / uvicorn**: FastAPI describes the app; uvicorn runs it and listens on port 8000.
- **Request/response model**: Pydantic classes that fix the shape of what comes in and goes out.
- **422**: the status code for "your request didn't match the model". Your code never ran, and no tokens were spent.
- **Session**: one conversation, named by `session_id`. The server keeps one history per session.
- **Streaming**: the answer in pieces, as the model writes it. Same total time, much shorter wait.
- **SSE**: the streaming format: `event: ...`, `data: ...`, then a blank line. Sent as `text/event-stream`.

## A4. How today works — two terminals, and the rules for ChatGPT

```
 TERMINAL 1 (server)                                   TERMINAL 2 (client)
 uv run python $env:WORK_REPO\s23\app.py              uv run python module03/s23_client.py
 ┌───────────────────────────────────────┐   HTTP     ┌───────────────────────────────┐
 │ YOUR app.py                           │◄───────────┤ POST /chat {session_id, msg}  │
 │  PART 1 (given): model, tools, loop   │            │                               │
 │  PART 2 (yours): endpoints, SESSIONS  ├───────────►│ {"answer": ..., "turns": 2}   │
 │        └──► Groq / Ollama             │  events    │ event: token / data: "Your"…  │
 └───────────────────────────────────────┘            └───────────────────────────────┘
        also: http://127.0.0.1:8000/docs in your browser - every endpoint, clickable
```

**The file has two parts.** PART 1 is Session 21's BankAssist, finished: the model, two tools, and your capped loop.
It now **streams** (`llm.stream` instead of `llm.invoke`) and catches bad tool calls (S20's Fix B). Read it once;
don't change it. PART 2 is yours: six TODOs, in order. After each one, run that section's check cell.

**Using ChatGPT is allowed, with the same three rules as Session 21:**
1. **Paste `module03/s23_version_card.md` first.** Without it, ChatGPT writes Flask, or uses streaming libraries
   that aren't installed.
2. **Never paste your `.env` or your API key.**
3. **Ask it to explain or fix your code, not to write the file.** At the exit check you'll explain one endpoint.

**Stuck?** Look at the **server** terminal (that's where your errors appear), read the last line, then find it in
`module03/s23_common_errors.md`.

---
# Part B · Build

## Section 1 · Set up your file
**What and why.** Your own copy of the starter, in your work repository: `s23/app.py`.

**What the next cell does.** 1) Finds the course repository root. 2) Loads your `.env`. 3) Copies
`s23_starter.py` to `<WORK_REPO>/s23/app.py`, **only if it isn't there already**, so re-running never wipes your
work. 4) Defines `show(...)`, which runs the checker on your file and prints one TODO's lines.

Before running it: `git pull` and `uv sync` in the course repository. There are two new packages today, `fastapi`
and `uvicorn`, both pinned.

In [ ]:
import os, sys, shutil, time
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "module02").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "module02"))
sys.path.insert(0, str(ROOT / "module03"))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=True)

WORK_REPO = Path(os.getenv("WORK_REPO") or r"C:\Users\you\Documents\genai-course-work")   # <-- your folder
target = WORK_REPO / "s23" / "app.py"
if target.exists():
    print("Already there - NOT overwritten (your work is safe):", target)
else:
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(ROOT / "module03" / "s23_starter.py", target)
    print("Created:", target)
print()
print("Now open it in VS Code:   code", target)
print("Model:", os.getenv("MODEL"), "  endpoint:", os.getenv("BASE_URL"))

from fastapi.testclient import TestClient
from s23_check import check_s23_work, load_student, parse_sse
def show(step):
    # Run the checker on YOUR file and print only the lines for one TODO.
    for r in check_s23_work(WORK_REPO):
        if step in r.name or (r.name.startswith(("work repo", "s23/", "your file")) and not r.passed):
            print(("PASS  " if r.passed else "FAIL  ") + r.name + (f"\n      -> {r.detail}" if r.detail else ""))

def live(fn):
    # Run a real-model demo; explain instead of crashing if the model can't be reached.
    try:
        fn()
    except NotImplementedError as e:
        print("not done yet:", e)
    except Exception as e:
        print(f"live demo skipped ({type(e).__name__}): is Ollama running / is your key in .env?")

**What you should see.** `Created: ...\s23\app.py` (or `Already there`), and your model.

**Read PART 1 now (5 minutes).** Find three things: `get_llm()` builds the model once, on first use. In `run_tool()`,
the `try/except` turns a bad tool call into an `INVALID:` message instead of a crash. That's S20's Fix B, and the
S21 homework. In `stream_turn()`, `llm_with_tools.stream(messages)` gives *pieces*, and `full = full + chunk` adds
them up into one whole message, so the loop can still check `full.tool_calls`. Everything else is your S21 loop.

## Section 2 · TODO 1 — the simplest endpoint: `GET /health`
**What and why.** Every service has a "are you alive?" endpoint. Load balancers call it every few seconds, Docker
calls it in Session 23, and you'll call it first in every demo. It needs no model, so it's the perfect first endpoint.

**The idea.** In FastAPI an endpoint is a normal function with a decorator. `@app.get("/health")` means "when a GET
request arrives for `/health`, run this function". Return a dict, and FastAPI sends it as JSON.

**What to type**: in `health()`, replace the `raise` line with:
```python
return {"status": "ok", "model": os.environ["MODEL"]}
```

In [ ]:
show("TODO 1")
mod = load_student(WORK_REPO)
client = TestClient(mod.app)          # real HTTP requests to your app - no server needed
try:
    r = client.get("/health")
    print()
    print("status code:", r.status_code)
    print("body:       ", r.json())
except NotImplementedError as e:
    print("not done yet:", e)

**What you should see.** `PASS TODO 1`, `status code: 200` and `{'status': 'ok', 'model': '...'}`.

**What just happened.** `TestClient` sent a real HTTP GET to your app *inside Python*, without starting a server.
Your checker uses the same trick, and in Mini-Project 1 your project's tests will too.

**Check yourself.** Why put the model's name in `/health`? (So whoever runs the service can see at a glance which
model is live. In a bank, "which model answered that customer?" is an audit question.)

## Section 3 · TODO 2 — the shape of a request and a reply
**What and why.** A public endpoint receives whatever people send: empty strings, a 50-page essay, a typo in a key.
Every message you pass on to the model costs tokens. So check the shape **at the door**.

**The idea.** A Pydantic model (Session 13's structured output, now on the way *in*) describes the request. FastAPI
checks every request against it. If it doesn't match, FastAPI replies **422** and your function never runs.

**What to type**:
- In `ChatRequest`, change the message line to: `message: str = Field(min_length=1, max_length=2000)`
- In `ChatResponse`, add two lines: `tool_steps: list[str]` and `turns: int`

In [ ]:
show("TODO 2")
mod = load_student(WORK_REPO)
from pydantic import ValidationError
for msg in ("What are your Saturday timings?", "", "x" * 5000):
    try:
        mod.ChatRequest(session_id="demo", message=msg)
        print(f"{msg[:30]!r:35} accepted")
    except ValidationError as e:
        print(f"{msg[:30]!r:35} REJECTED: {e.errors()[0]['msg']}")
print("ChatResponse fields:", list(mod.ChatResponse.model_fields))

**What you should see.** The question is accepted. The empty message and the 5,000-character one are rejected.
`ChatResponse` lists `session_id, answer, tool_steps, turns`.

**Check yourself.** Is `max_length=2000` a security control or a cost control? (Both. It caps what one request can
cost, and it limits how much room an attacker has for a prompt injection. S18 showed how much a long message can
hide.)

## Section 4 · TODO 3 — memory, trimmed: `window()`
**What and why.** Session 19 measured it: resending the whole conversation makes every call cost more than the last,
and a sliding window keeps the cost flat. Your server will be talking to many customers at once, so the window
isn't optional.

**The idea.** One turn is two messages (the customer's `HumanMessage`, BankAssist's `AIMessage`). Keep the last
`keep_turns` turns, which means the last `2 * keep_turns` messages. The system prompt isn't in `history`; it's
added fresh on every call.
```
 history:  q0 a0 q1 a1 q2 a2 q3 a3 q4 a4 q5 a5      keep_turns = 4
 window:               q2 a2 q3 a3 q4 a4 q5 a5      ← q0 and q1 are forgotten, on purpose
```

**What to type**: in `window(...)`, replace the `raise` line with:
```python
return history[-2 * keep_turns:] if keep_turns > 0 else []
```
(Why the `if`? Because `history[-0:]` is the **whole** list in Python, not an empty one.)

In [ ]:
show("TODO 3")
mod = load_student(WORK_REPO)
from langchain_core.messages import HumanMessage, AIMessage
history = []
for i in range(6):
    history += [HumanMessage(f"q{i}"), AIMessage(f"a{i}")]
try:
    print("history:", " ".join(m.content for m in history))
    print("window :", " ".join(m.content for m in mod.window(history)), f"  (KEEP_TURNS = {mod.KEEP_TURNS})")
except NotImplementedError as e:
    print("not done yet:", e)

**What you should see.** `window : q2 a2 q3 a3 q4 a4 q5 a5`.

**Check yourself.** A customer gives their income at turn 1 and asks about eligibility at turn 7. What happens?
(The window has dropped turn 1, so the model doesn't know the income. It asks again, or worse, guesses. That's
S19's "silent forgetting". In Mini-Project 1 you'll pick window or summary and defend the choice with numbers.)

## Section 5 · TODO 4 — `POST /chat`, with memory
**What and why.** This is the endpoint a website would call. It takes `{session_id, message}`, finds that session's
history, sends the model a window of it plus the new message, saves the turn, and replies.

**The idea.**
```
 POST /chat {"session_id": "priya", "message": "What's my name?"}
   history  = SESSIONS["priya"]                       (created empty the first time)
   messages = [SYSTEM] + window(history) + [new question]
   answer   = run_turn(...)                           (PART 1 - your S21 loop)
   history += [question, answer]                      (save BOTH, or the next turn can't see this one)
   reply      {"answer": ..., "tool_steps": [...], "turns": 2}
```
**The S19 lesson, one level up.** The API is still stateless. "The assistant remembers" is now the **server**
resending `SESSIONS[session_id]`. Only the question and the final answer are saved, not the tool messages, so
history stays small.

**What to type**: replace the `raise` line in `chat(...)` with the code in its docstring.

In [ ]:
show("TODO 4")
mod = load_student(WORK_REPO)
client = TestClient(mod.app)
def demo():
    for sid, msg in (("priya", "Hi, my name is Priya and I earn 80,000 a month."),
                     ("priya", "What is my name, and what do I earn?"),
                     ("ravi",  "What is my name?")):
        t = time.time()
        r = client.post("/chat", json={"session_id": sid, "message": msg})
        r.raise_for_status()
        body = r.json()
        print(f"[{sid}] you : {msg}")
        print(f"[{sid}] bank: {body['answer']}")
        print(f"        tools: {body['tool_steps'] or 'none'}   turns: {body['turns']}   {time.time() - t:.1f}s\n")
    print("SESSIONS on the server:", {k: len(v) for k, v in mod.SESSIONS.items()}, "(messages per session)")
live(demo)

**What you should see.** Two `PASS TODO 4` lines. Priya's second answer knows her name and income, Ravi's doesn't,
and `SESSIONS` shows `{'priya': 4, 'ravi': 2}`.

**Two things to notice, on a small model especially.**
- It may call `get_rate` when nobody asked for a rate. That's S20's over-calling problem, and you'll measure it in
  Mini-Project 1.
- Ravi's reply may say *"you mentioned earlier..."*, but Ravi's list is empty. The model **invented** a memory.
  That's not a leak between sessions. Always check `SESSIONS`, not the model's claims.

**Check yourself.** Two customers both send `session_id: "1"`. What happens? (They share one memory, and each sees
the other's data. Whoever creates `session_id` must make it unique and unguessable. In a real app that's the login
system, not the customer.)

## Section 6 · TODO 5 — stream it, with Server-Sent Events
**What and why.** Your `/chat` makes the customer wait for the whole answer. With streaming, the first words appear
as soon as the model writes them.

**The idea.** SSE is plain text over HTTP. Each event is an `event:` line, a `data:` line, and a **blank line**:
```
event: tool
data: {"text": "calculate_emi({...}) -> 21696"}

event: token
data: {"text": "Your "}

event: done
data: {"text": "Your EMI is about Rs 21,696 a month."}

```
`StreamingResponse` sends each string your generator `yield`s the moment it's yielded. `json.dumps` on the data
matters: a token with a newline in it would otherwise end the event early.

**What to type**: two parts. 5a: in `sse(...)`, `return f"event: {event}\ndata: {json.dumps(data)}\n\n"`.
5b: replace the `raise` in `chat_stream(...)` with the code in its docstring. Note *when* the turn is saved: on the
`done` event, after the customer has seen the whole answer.

In [ ]:
show("TODO 5")
mod = load_student(WORK_REPO)
client = TestClient(mod.app)
def demo():
    print("--- the raw stream, exactly as a browser receives it (first 500 characters) ---")
    r = client.post("/chat/stream", json={"session_id": "s", "message": "What is the EMI on a 25 lakh home loan at 8.5% for 20 years?"})
    print(r.text[:500])
    events = parse_sse(r.text)
    print("--- events:", len(events), "  kinds:", sorted({k for k, _ in events}))

    print("\n--- why stream? time to the FIRST word vs the WHOLE answer ---")
    from langchain_core.messages import SystemMessage, HumanMessage
    t, first = time.time(), None
    for kind, text in mod.stream_turn(mod.get_llm(), [SystemMessage(mod.SYSTEM), HumanMessage("What documents do I need for a home loan, and how long does approval take?")]):
        if kind == "token" and first is None:
            first = time.time() - t
    print(f"first word after {first:.1f}s, whole answer after {time.time() - t:.1f}s")
live(demo)

**What you should see.** Two `PASS TODO 5` lines. In the raw stream, one `event: tool` block, then many
`event: token` blocks, then `event: done`. Then the timing: the first word arrives well before the whole answer
(on Groq maybe 0.5s vs 2s; on a laptop model maybe 2s vs 10s).

**Check yourself.** Does streaming make BankAssist faster? (No. The total time is the same. It makes the *wait*
shorter, which is what customers feel. That's the difference between latency to the first token and total latency.)

## Section 7 · TODO 6 (stretch) — let a customer say "forget me"
**What and why.** Customers (and India's DPDP Act) can ask you to delete what they told you. Add
`DELETE /sessions/{session_id}`. It removes that session from `SESSIONS` and returns `{"deleted": True}`, or
`{"deleted": False}` if there was nothing to delete.

**Hint.** `@app.delete("/sessions/{session_id}")` on a function that takes `session_id: str`. `SESSIONS.pop(key, None)`
removes a key without crashing if it's missing. Try writing it without looking anything up. It's three lines.

In [ ]:
show("TODO 6")

## Section 8 · Run it for real — two terminals, a browser, a client
Everything so far used `TestClient`. Now run a real server, the way it will run in Docker in Session 23b.

1. **Terminal 1** (VS Code terminal, course repo folder): `uv run python $env:WORK_REPO\s23\app.py`
   You should see `Uvicorn running on http://127.0.0.1:8000`. Leave it running.
2. **Browser**: open **http://127.0.0.1:8000/docs**. FastAPI wrote this page from your code and your Pydantic
   models. Click `POST /chat` → *Try it out* → send a message. Then send `"message": ""` and watch the 422.
3. **Terminal 2** (click `+` in the VS Code terminal panel, course repo folder): `uv run python module03/s23_client.py`
   It tours your API: memory, two sessions, a 422, and a streamed answer with a tool. Watch the words arrive.
4. Look back at **Terminal 1**: one log line per request, with its status code.
5. **One question of your own**: `uv run python module03/s23_client.py "Can I prepay my floating-rate home loan?"`

**Every time you change `app.py`: Ctrl+C in Terminal 1, then start it again.** The server runs the code it loaded at
startup.

The cell below only checks whether your server is up (it's harmless if it isn't).

In [ ]:
import httpx
try:
    print(httpx.get("http://127.0.0.1:8000/health", timeout=3).json())
except httpx.ConnectError:
    print("No server on port 8000 - start it in Terminal 1:  uv run python $env:WORK_REPO\\s23\\app.py")

## Section 9 · Break it on purpose: three honest limits
**What and why.** Your server works. Before you put it in a container in Session 23, see where it breaks. Each of
these is a real production problem with a real fix, and none of them is a bug in your code.

**What the next cell does.** 1) A conversation longer than the window: does turn 1 survive? 2) A "restart": it
reloads your file, which is exactly what Ctrl+C plus starting again does. 3) What a 422 costs you.

In [ ]:
mod = load_student(WORK_REPO)
client = TestClient(mod.app)
def demo():
    sid = "longchat"
    client.post("/chat", json={"session_id": sid, "message": "My reference number is NW-4471. Please remember it."})
    for q in ("What are your Saturday timings?", "Is there a prepayment penalty on floating loans?",
              "What is the minimum credit score?", "What is the home loan fixed rate?", "Thanks!"):
        client.post("/chat", json={"session_id": sid, "message": q})
    r = client.post("/chat", json={"session_id": sid, "message": "What was my reference number?"}).json()
    print(f"1) after {r['turns'] - 1} turns (window = {mod.KEEP_TURNS}):", r["answer"][:160])

    mod2 = load_student(WORK_REPO)        # what Ctrl+C + restart does to SESSIONS
    print("2) sessions before restart:", list(mod.SESSIONS), " after restart:", list(mod2.SESSIONS))

    bad = client.post("/chat", json={"session_id": sid, "message": ""})
    print("3) empty message ->", bad.status_code, "- your code never ran, zero tokens spent")
live(demo)

**What you should see, and what it means.**

| Limit | What you saw | The real fix (and where you'll learn it) |
|---|---|---|
| **The window forgets** | NW-4471 is gone after more than 4 turns. The model says it doesn't know, or invents one | Summary memory (S19), or saving key facts separately; Mini-Project 1 asks you to choose |
| **RAM forgets everything** | A restart empties `SESSIONS`. Two copies of the server wouldn't share it either | A database or Redis behind `SESSIONS` (the agents module: LangGraph checkpoints) |
| **No one is logged in** | Anyone who guesses `"longchat"` reads that conversation | Authentication; `session_id` comes from the login, not the client (the security module) |

**Check yourself.** Your manager asks "can we run three copies of this server behind a load balancer?" What breaks?
(Each copy has its own `SESSIONS`, so a customer whose second message reaches a different copy is a stranger
again. Memory has to move out of RAM first.)

## Section 10 · Bridge (15 minutes): LiteLLM from S22 — where it plugs into this server
**The problem it solves.** Your `.env` switches between Groq and Ollama because both speak the OpenAI format. Many
providers don't (Anthropic, Gemini's own API, Bedrock), and a production service wants more: **fallbacks** (Groq is
rate-limited, so try another provider), **cost tracking** per request, and one place to hold all the keys.

**What LiteLLM is.** A library that gives about 100 providers one call format, plus an optional **gateway** server
your apps call instead of calling providers directly:
```python
# you ran this in S22 - here, read it, don't run it
from litellm import completion
r = completion(model="groq/openai/gpt-oss-20b", messages=[...],
               fallbacks=["ollama/llama3.2:3b"])      # if Groq fails, try the laptop model
print(r.choices[0].message.content, r._hidden_params["response_cost"])
```
**Where it would go in your app.** Only inside `make_llm()`. Your endpoints, sessions and SSE wouldn't change at all.
That's why it's a sidebar: it's a swap of the model layer, not a new way to build.

**Why today's server doesn't use it yet.** For two providers that both speak the OpenAI format, `ChatOpenAI` plus
`.env` is enough. Next session (S23b) you run LiteLLM as a **gateway server**: then `make_llm()` only changes its
`base_url` to the gateway, and the fallbacks, keys and cost logs live there for every app at once. Mini-Project 1
(ShopAssist) is built exactly that way.

**Interview line.** "LangChain is how I build the app; LiteLLM is how I'd talk to many providers through one
interface, with fallbacks and cost tracking. They solve different problems and can be used together."

## Section 11 · Checkpoint
**What and why.** `check_s23_work` imports **your** file, swaps in a scripted model, and sends real HTTP requests to
your app with `TestClient`. It costs nothing, and it's the same check `pytest` runs.

In [ ]:
from prompt_utils import results_table
print(results_table(check_s23_work(WORK_REPO)))

## Section 12 · Commit and push
Every line `PASS` (TODO 6 says "not attempted - optional" if you skipped it, which still passes). Then:
```
cd $env:WORK_REPO
git add s23/app.py
git commit -m "Session 23: BankAssist served with FastAPI + SSE"
git push
cd <your course repo folder>
uv run pytest tests/test_s23.py -rs
```

---
# Part C · Wrap-up

## C1. Summary
- An **endpoint** is a function with a decorator. **FastAPI** describes the app, and **uvicorn** runs it.
- **Pydantic** request models reject bad input with a **422** before it costs a token.
- The API is still stateless. The **server** now keeps one history per `session_id` and sends a **window** of it.
- **Streaming** (`llm.stream`) shortens the *wait*, not the total time. **SSE** is `event:` + `data:` + a blank line.
- `TestClient` tests an API without a server, which is how your checker (and your Mini-Project 1 tests) work.
- **LiteLLM** (S22) is a model layer (many providers, fallbacks, cost). As a gateway (S23b) it only changes `make_llm()`'s `base_url`.

## C2. Honest limits
- **Memory in RAM** is lost on restart and not shared between copies of the server.
- **`session_id` from the client is not security.** Anyone who guesses it reads the conversation.
- **The window forgets silently.** A fact from turn 1 is gone after `KEEP_TURNS` turns, and the model may invent it.
- **Plain `def` endpoints use a thread pool.** That's fine for a class, but at high traffic you'd use `async` and an
  async model client.
- **Your API key is on the server.** That's correct: it never goes to the browser. But the server must never log it
  or put it in a container image (Session 23b).

## C3. Interview questions, answered
1. **How do you serve an LLM app?** A web framework (FastAPI) behind a server (uvicorn), Pydantic models for input
   and output, per-session memory with a window, streaming for perceived latency, a `/health` endpoint, and a
   container to ship it.
2. **Where does conversation memory live in your service?** On the server, keyed by session. In a demo that's a dict
   in RAM. In production it's a database or Redis, so it survives restarts and is shared across instances.
3. **Why stream, and how?** It cuts time to the first token, which is what users feel. I use the model's streaming
   API and send Server-Sent Events: `text/event-stream`, one `event:`/`data:` block per piece.
4. **SSE or WebSockets?** SSE for one-way server-to-client streams over plain HTTP. It's simple, works through
   proxies, and has a built-in browser API. WebSockets when both sides must send at any time (voice, collaboration).
5. **How do you stop an expensive or malicious request?** Validate at the door (lengths, required fields → 422), cap
   tool steps, window the history, rate-limit per user, and log tokens per request.
6. **LangChain vs LiteLLM?** LangChain builds the app (prompts, tools, chains). LiteLLM unifies provider calls
   (one format, fallbacks, cost). They solve different problems and work together.

## C4. Common mistakes
See `module03/s23_common_errors.md`. The top three: forgetting to restart the server after a change, saving only the
question (so memory never works), and opening `POST /chat` in a browser (405).

## C5. Homework (the first one is required for Session 23b)
1. **Required: install Docker Desktop and prove it works.** docs.docker.com/desktop/setup/install/windows-install/
   → install (accept WSL 2), restart, open Docker Desktop once, then in a new terminal:
   `docker run hello-world`. It must print *"Hello from Docker!"*. **Session 23b can't start for you without it**, and
   the first Windows install often needs a restart and a BIOS virtualisation setting, so don't leave it to the
   morning of Session 23b. Mac: install Docker Desktop for Mac. Linux: `sudo apt install docker.io`.
   **Already have Docker Desktop, but `docker` is "not recognized"?** The install is half-finished (often an
   auto-update cut off by a restart): reinstall with `winget install -e --id Docker.DockerDesktop`, restart, and open
   Docker Desktop once. More in `module03/s23b_common_errors.md`.
2. **Read Part A of `module03/s23b_lab.md`** (10 minutes): next session runs the LiteLLM gateway from S22 in front
   of a server like yours, in Docker.
3. TODO 6, if you skipped it.
4. **Token log.** In `chat()`, print the number of messages sent and the length of the answer for each request. Then
   send 10 messages in one session: does the line stay flat (the window working) or grow?